# 03 — Decomposing Time Series: Separating Signal from Noise

**Companion notebook to Article 3 of the *Time Series Feature Engineering* series.**

📖 Read the article: [Medium](https://medium.com/@asidd24/decomposing-time-series-separating-signal-from-noise-f76b05f2bc9f?postPublishedType=initial) · [LinkedIn Newsletter](https://www.linkedin.com/newsletters/time-series-engineered-7485187061080137728/)

---

This notebook demonstrates:
1. **Additive vs Multiplicative** seasonality visual check
2. **Classical Decomposition** using statsmodels
3. **STL (Seasonal-Trend Decomposition using LOESS)**
4. **MSTL (Multiple Seasonal-Trend Decomposition using LOESS)**
5. **LOWESS Smoothing** to isolate trend
6. **Decomposition features** for ML models (leakage-safe)
7. **STL-based Anomaly Detection** using residuals and 3-sigma thresholds

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.seasonal import seasonal_decompose, STL, MSTL
from statsmodels.nonparametric.smoothers_lowess import lowess
from src.data import generate_daily_sales

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True
np.random.seed(42)

## 1. Additive vs Multiplicative Seasonality

- **Additive model**: $y(t) = Trend(t) + Seasonality(t) + Residual(t)$
- **Multiplicative model**: $y(t) = Trend(t) \times Seasonality(t) \times Residual(t)$

In [ ]:
# Generate additive vs multiplicative data
t = np.arange(365)
trend = 50 + 0.5 * t
seasonal_add = 10 * np.sin(2 * np.pi * t / 30)
seasonal_mult = (1 + 0.15 * np.sin(2 * np.pi * t / 30))

add_series = trend + seasonal_add + np.random.normal(0, 2, 365)
mult_series = trend * seasonal_mult + np.random.normal(0, 2, 365)

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
axes[0].plot(add_series, label='Additive Series', color='royalblue')
axes[0].plot(trend, '--', color='orange', label='Trend')
axes[0].set_title('Additive: Constant Seasonal Amplitude')
axes[0].legend()

axes[1].plot(mult_series, label='Multiplicative Series', color='forestgreen')
axes[1].plot(trend, '--', color='orange', label='Trend')
axes[1].set_title('Multiplicative: Expanding Seasonal Amplitude')
axes[1].legend()

plt.tight_layout()
plt.show()

## 2. Classical Decomposition

Let's generate daily sales and apply classical additive decomposition using moving averages.

In [ ]:
# Generate sales with weekly seasonality
dates = pd.date_range('2022-01-01', periods=730, freq='D')
t = np.arange(730)
trend = 100 + 0.05 * t
seasonality = 15 * np.sin(2 * np.pi * t / 7)  # weekly cycle
noise = np.random.normal(0, 5, 730)
sales = pd.Series(trend + seasonality + noise, index=dates, name='sales')

# Classical additive decomposition
result_classical = seasonal_decompose(sales, model='additive', period=7)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
result_classical.observed.plot(ax=axes[0], title='Observed')
result_classical.trend.plot(ax=axes[1], title='Trend')
result_classical.seasonal.plot(ax=axes[2], title='Seasonality')
result_classical.resid.plot(ax=axes[3], title='Residual')
plt.tight_layout()
plt.show()

## 3. STL — Seasonal-Trend Decomposition Using LOESS

STL handles changing seasonality, doesn't leave endpoint gaps, and is robust to outliers.

In [ ]:
# Run STL decomposition
stl = STL(sales, period=7, robust=True)
result_stl = stl.fit()

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
result_stl.observed.plot(ax=axes[0], title='Observed')
result_stl.trend.plot(ax=axes[1], title='STL Trend')
result_stl.seasonal.plot(ax=axes[2], title='STL Seasonality')
result_stl.resid.plot(ax=axes[3], title='STL Residual')
plt.tight_layout()
plt.show()

## 4. Multiple Seasonalities — MSTL

Let's model daily data containing **both weekly and yearly** seasonality.

In [ ]:
# Generate 3 years of daily sales with weekly and yearly seasonality
dates_long = pd.date_range('2022-01-01', periods=1095, freq='D')
t_long = np.arange(1095)
weekly = 15 * np.sin(2 * np.pi * t_long / 7)
yearly = 30 * np.sin(2 * np.pi * t_long / 365.25)
trend_long = 100 + 0.05 * t_long
noise_long = np.random.normal(0, 5, 1095)
sales_multi = pd.Series(trend_long + weekly + yearly + noise_long, index=dates_long)

# MSTL decomposition
mstl = MSTL(sales_multi, periods=[7, 365])
result_mstl = mstl.fit()

fig, axes = plt.subplots(5, 1, figsize=(14, 12), sharex=True)
result_mstl.observed.plot(ax=axes[0], title='Observed')
result_mstl.trend.plot(ax=axes[1], title='Trend')
result_mstl.seasonal['seasonal_7'].plot(ax=axes[2], title='Weekly Seasonality')
result_mstl.seasonal['seasonal_365'].plot(ax=axes[3], title='Yearly Seasonality')
result_mstl.resid.plot(ax=axes[4], title='Residual')
plt.tight_layout()
plt.show()

## 5. LOWESS Smoothing

When you only want to smooth out the noise and capture the trend.

In [ ]:
# Extract just the trend using LOWESS
smoothed = lowess(sales.values, np.arange(len(sales)), frac=0.1, return_sorted=False)
trend_series = pd.Series(smoothed, index=sales.index)

fig, ax = plt.subplots(figsize=(14, 4))
sales.plot(ax=ax, label='Raw', alpha=0.5)
trend_series.plot(ax=ax, label='LOWESS Trend', linewidth=2, color='crimson')
ax.legend()
ax.set_title('LOWESS Smoothing (frac=0.1)')
plt.tight_layout()
plt.show()

## 6. Decomposition Features (Leakage-Safe)

Using trend and residuals as model inputs. Note: always recompute inside your CV loops to avoid leakage!

In [ ]:
stl_fit = STL(sales, period=7, robust=True).fit()

features = pd.DataFrame({
    'target': sales,
    'stl_trend': stl_fit.trend,
    'stl_seasonal': stl_fit.seasonal,
    'stl_residual': stl_fit.resid
})

# Extracting lag features from residuals (safe rolling volatility)
features['residual_volatility'] = stl_fit.resid.shift(1).rolling(7).std()
features.dropna().head(10)

## 7. Anomaly Detection

Identifying values deviating beyond $\pm 3\sigma$ on decomposition residuals.

In [ ]:
# STL-based anomaly detection
stl_fit = STL(sales, period=7, robust=True).fit()
residuals = stl_fit.resid

threshold_upper = residuals.mean() + 3 * residuals.std()
threshold_lower = residuals.mean() - 3 * residuals.std()

anomalies = sales[(residuals > threshold_upper) | (residuals < threshold_lower)]
print(f"Detected {len(anomalies)} anomalies out of {len(residuals)} observations")
print(f"Anomaly rate: {len(anomalies) / len(residuals):.2%}")

# Visualization
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

sales.plot(ax=ax1, label='Sales', alpha=0.7)
ax1.scatter(anomalies.index, anomalies.values, color='red', label='Anomaly', zorder=5)
ax1.set_title('Raw Sales with Detected Anomalies')
ax1.legend()

residuals.plot(ax=ax2, label='STL Residuals', color='purple')
ax2.axhline(threshold_upper, color='red', linestyle='--', label='+3 Sigma')
ax2.axhline(threshold_lower, color='red', linestyle='--', label='-3 Sigma')
ax2.fill_between(residuals.index, threshold_lower, threshold_upper, color='green', alpha=0.1, label='Normal Range')
ax2.set_title('STL Residuals with Outlier Thresholds')
ax2.legend()

plt.tight_layout()
plt.show()